# J. Evaluation Metrics

- **J1**: Train + Val loss recorded
- **J2**: Train + Val accuracy recorded
- **J3**: Confusion matrix
- **J4**: Precision, Recall, F1
- **J5**: Learning curve

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from network import Network, NetworkConfig
from optimizer import Adam

## Setup: Train a Model

In [ ]:
np.random.seed(42)

n = 200
x = np.random.randn(n, 30)
y = np.zeros((n, 2))
y[np.arange(n), np.random.randint(0, 2, n)] = 1

split = int(0.8 * n)
x_tr, x_val = x[:split], x[split:]
y_tr, y_val = y[:split], y[split:]

mean = x_tr.mean(axis=0); std = x_tr.std(axis=0) + 1e-8
x_tr = (x_tr - mean) / std
x_val = (x_val - mean) / std

config = NetworkConfig(layers=[30, 24, 24, 24, 2], activation='relu',
    loss='cross_entropy', output_activation='softmax', weights_initializer='heUniform')
net = Network(config)
opt = Adam(learning_rate=0.001)

history = {'loss':[], 'val_loss':[], 'accuracy':[], 'val_accuracy':[]}

for epoch in range(200):
    idx = np.arange(len(x_tr)); np.random.shuffle(idx)
    for i in range(0, len(x_tr), 8):
        net.forward(x_tr[idx[i:i+8]])
        nw, nb = net.backward(y_tr[idx[i:i+8]])
        opt.update(net, nw, nb)

    tr_out = net.forward(x_tr)
    history['loss'].append(net.loss(y_tr, tr_out))
    history['accuracy'].append(np.mean(np.argmax(tr_out, axis=1) == np.argmax(y_tr, axis=1)))

    v_out = net.forward(x_val)
    history['val_loss'].append(net.loss(y_val, v_out))
    history['val_accuracy'].append(np.mean(np.argmax(v_out, axis=1) == np.argmax(y_val, axis=1)))

print(f'Final train: loss={history["loss"][-1]:.4f} acc={history["accuracy"][-1]:.4f}')
print(f'Final val:   loss={history["val_loss"][-1]:.4f} acc={history["val_accuracy"][-1]:.4f}')

## J1 & J2: History Recorded

In [ ]:
for key in ['loss', 'val_loss', 'accuracy', 'val_accuracy']:
    print(f'{key:>15}: {len(history[key])} entries')

ok = all(len(history[k]) == 200 for k in history)
print(f'Status: {"PASS" if ok else "FAIL"}')

## J3: Confusion Matrix

In [ ]:
v_out = net.forward(x_val)
y_pred = np.argmax(v_out, axis=1)
y_true = np.argmax(y_val, axis=1)

labels = ['B (Benign)', 'M (Malignant)']
cm = np.zeros((2, 2), dtype=int)
for t, p in zip(y_true, y_pred):
    cm[t][p] += 1

print('Confusion Matrix:')
print(f'{"":>15} | {"Pred B":>8} | {"Pred M":>8}')
print('-' * 40)
for i in range(2):
    print(f'{labels[i]:>15} | {cm[i][0]:>8} | {cm[i][1]:>8}')

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(labels); ax.set_yticklabels(labels)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i][j]), ha='center', va='center', fontsize=16,
                color='white' if cm[i][j] > cm.max()/2 else 'black')
plt.colorbar(im); plt.tight_layout(); plt.show()

## J4: Precision, Recall, F1-Score

In [ ]:
def metrics(y_true, y_pred, n_classes):
    result = {}
    for c in range(n_classes):
        tp = np.sum((y_pred == c) & (y_true == c))
        fp = np.sum((y_pred == c) & (y_true != c))
        fn = np.sum((y_pred != c) & (y_true == c))
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        result[c] = {'precision': prec, 'recall': rec, 'f1': f1, 'support': int(np.sum(y_true == c))}
    return result

m = metrics(y_true, y_pred, 2)
labels = ['B (Benign)', 'M (Malignant)']

print(f'{"Class":>15} | {"Precision":>10} | {"Recall":>10} | {"F1":>10} | {"Support":>8}')
print('-' * 62)
for c in range(2):
    print(f'{labels[c]:>15} | {m[c]["precision"]:>10.4f} | {m[c]["recall"]:>10.4f} | {m[c]["f1"]:>10.4f} | {m[c]["support"]:>8}')

macro_f1 = np.mean([m[c]['f1'] for c in range(2)])
print('-' * 62)
print(f'{"Macro Avg":>15} | {np.mean([m[c]["precision"] for c in range(2)]):>10.4f} | {np.mean([m[c]["recall"] for c in range(2)]):>10.4f} | {macro_f1:>10.4f} |')

## J5: Learning Curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history['loss']) + 1)

ax1.plot(epochs, history['loss'], label='Train')
ax1.plot(epochs, history['val_loss'], label='Val')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(True)

ax2.plot(epochs, history['accuracy'], label='Train')
ax2.plot(epochs, history['val_accuracy'], label='Val')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.legend(); ax2.grid(True)

plt.tight_layout(); plt.show()
print('Learning curve plotted successfully.')